# Lab 1 — Pydantic Validation for AI Request/Response Contracts

Difficulty: Beginner | ~30-35 min | No prerequisites

### Step 1: Install Dependencies

We start by installing our target libraries: `fastapi` for the API framework, `pydantic` for schema validation, and `httpx` as the HTTP client that powers FastAPI's `TestClient`.

In [ ]:
# Install the exact pinned versions of the required libraries.
!pip install fastapi pydantic httpx


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Python Imports

Next, we import the standard typing utilities and the necessary classes from FastAPI and Pydantic that we will use to build the application.

In [91]:
# Import standard typing and FastAPI helpers.
from typing import Literal
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, ValidationError

### Step 3: Define Mock Replies

We define a mock database of LLM outputs. This simulates what an LLM would return. We have a valid reply for France and a reply for Germany which has empty `content` field, simulating an unpredictable LLM error.

In [92]:
# Define pre-written replies for our mock LLM.
MOCK_REPLIES = {
    "What's the capital of France?": {
        "role": "assistant",
        "content": "The capital of France is Paris."
    },
    "What's the capital of Germany?": {
        "role": "assistant",
        "content": ""
    }
}

### Step 4: Mock LLM Lookup Function


In [93]:
# Mock function called multiple times to simulate LLM replies.
def mock_llm_reply(user_message):
    """Simulates an LLM response by querying a local dictionary lookup."""

    reply = MOCK_REPLIES.get(user_message)

    return reply

### Step 5: Message Schema

We define the `Message` class using Pydantic's `BaseModel`. This validates that conversation messages have valid roles ('user', 'assistant') and text content.

In [94]:
# Define the message schema containing role and content.
class Message(BaseModel):
    role: Literal["assistant", "user"]
    content: str = Field(min_length= 1)

### Step 6: ChatRequest Schema

The `ChatRequest` class validates the payload sent by the user to our API, ensuring temperature is between 0.0 and 1.0, and max_tokens is greater than 0.

In [95]:
# Define incoming request schema with validation bounds.
class ChatRequest(BaseModel):
    messages: Message
    temperature: float = Field(ge=0.0, le=1.0)
    max_tokens: int = Field(gt=0)

### Step 7: ChatResponse Schema

The `ChatResponse` class validates the final validated output that will be returned to the client. This guards against structural errors generated by the LLM (like empty response).

In [96]:
# Define outgoing response schema.
class ChatResponse(BaseModel):
    role: str
    content: str = Field(min_length=10)

### Step 8: FastAPI Endpoint with Symmetrical Validation

We initialize our FastAPI app and implement the `/chat` route. The route validates request payloads automatically using `ChatRequest`. Within the endpoint, we fetch the response from `mock_llm_reply` and validate it against `ChatResponse`. If the LLM response is malformed, we catch the `ValidationError` and return a structured 500 error.

In [97]:
# Initialize FastAPI application and POST chat endpoint.
app = FastAPI()

@app.post("/chat")
def chat_endpoint(request: ChatRequest):

    user_message = request.messages.content
    
    # Get raw dictionary response from mock LLM.
    reply = mock_llm_reply(user_message)

    try:
        # Validate reply dict against output schema.
        return ChatResponse(**reply)
    except ValidationError as e:
        # Return standard 500 error structure on validation failure.
        return JSONResponse(
            status_code = 500,
            content = {"error": "response_validation_failed", "detail": e.errors()[0]['msg']}
        )

### Step 9: Initialize TestClient

We create a TestClient instance to test the API endpoint inside the notebook environment.

In [98]:
# Instantiating TestClient to run local API requests.
client = TestClient(app)

### Step 10: Demonstration Case 1 — Valid Request, Valid Response

We send a valid request payload to the `/chat` endpoint. The model outputs a correct reply structure. The request is processed successfully (HTTP 200 OK).

In [99]:
# Case 1: Valid request, valid response (expecting 200 OK).
payload_1 = {
    "messages":
        {"role": "user", "content": "What's the capital of France?"},
    "temperature": 0.7, "max_tokens": 150
}
response_1 = client.post("/chat", json=payload_1)
print("Status Code:", response_1.status_code)
print("Response JSON:", response_1.json())

Status Code: 200
Response JSON: {'role': 'assistant', 'content': 'The capital of France is Paris.'}


### Step 11: Demonstration Case 2 — Invalid Request (Bad Temperature)

We send a payload with temperature=5.0. Since this exceeds our Field definition constraint (le=1.0), FastAPI automatically catches the error and returns HTTP 422 Unprocessable Entity.

In [100]:
# Case 2: Invalid request (temperature out of bounds, expecting 422).
payload_2 = {
    "messages": {"role": "user", "content": "What's the capital of France?"},
    "temperature": 5.0, "max_tokens": 150
}
response_2 = client.post("/chat", json=payload_2)
print("Status Code:", response_2.status_code)
print("Response JSON:", response_2.json())

Status Code: 422
Response JSON: {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'temperature'], 'msg': 'Input should be less than or equal to 1', 'input': 5.0, 'ctx': {'le': 1.0}}]}


### Step 12: Demonstration Case 3 — Invalid Request (Missing Required Field)

We send a payload missing the mandatory `messages` field. FastAPI auto-rejects the request with HTTP 422 before calling our endpoint logic.

In [101]:
# Case 3: Invalid request (missing required messages field, expecting 422).
payload_3 = {"temperature": 0.7, "max_tokens": 150}
response_3 = client.post("/chat", json=payload_3)
print("Status Code:", response_3.status_code)
print("Response JSON:", response_3.json())

Status Code: 422
Response JSON: {'detail': [{'type': 'missing', 'loc': ['body', 'messages'], 'msg': 'Field required', 'input': {'temperature': 0.7, 'max_tokens': 150}}]}


### Step 13: Demonstration Case 4 — Valid Request, Malformed Response

We query about the capital of Germany. The request is valid, but the mock LLM returns an empty content field. Our endpoint's response validation catches this error and returns a clean HTTP 500 with our structured error message.

In [102]:
# Case 4: Valid request, malformed response (expecting 500).
payload_4 = {
    "messages": {"role": "user", "content": "What's the capital of Germany?"},
    "temperature": 0.7, "max_tokens": 150
}
response_4 = client.post("/chat", json=payload_4)
print("Status Code:", response_4.status_code)
print("Response JSON:", response_4.json())

Status Code: 500
Response JSON: {'error': 'response_validation_failed', 'detail': 'String should have at least 10 characters'}
